# Odlar-1.5: Colab-da təlim

Odlar-1-dən 3 dəfə böyük model (~40M parametr), kontekst 512 token, data: Vikipediya + FineWeb-2.

1. Yuxarıda **Runtime → Change runtime type → T4 GPU** seç → **Save**
2. **Runtime → Run all** bas
3. Google Drive icazəsi soruşanda **icazə ver**: model və irəliləyiş `MyDrive/Odlar15` qovluğuna yazılır
4. Səhifəni açıq saxla. Sessiya kəsilsə, sadəcə yenidən **Run all** bas: təlim qaldığı yerdən davam edəcək

Sonda `MyDrive/Odlar15/odlar15.zip` hazır olacaq.

In [ ]:
# 1) Ayarlar
WIKI_CHARS     = 300_000_000     # Vikipediyanın hamısı
FINEWEB_CHARS  = 1_200_000_000   # FineWeb-2-dən nə qədər mətn götürək
TOK_TRAIN_CHARS = 500_000_000    # tokenizer bu qədər mətn üzərində öyrənir
VOCAB_SIZE  = 16000
BLOCK_SIZE  = 512
N_LAYER, N_HEAD, N_EMBD = 10, 8, 512   # ~40M parametr
BATCH_SIZE  = 32
LR          = 5e-4
MAX_STEPS   = 15_000                    # 15,000 x 32 x 512 = ~245M token
EVAL_EVERY  = 500
SAMPLE_PROMPT = "Bakı şəhəri"

import os, subprocess
try:
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("GPU tapılmadı! Runtime → Change runtime type → T4 GPU")

# Google Drive: bütün nəticələr burada saxlanır
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/Odlar15"
except ImportError:
    OUT = "./odlar15_out"
os.makedirs(OUT, exist_ok=True)
print("Nəticələr buraya yazılacaq:", OUT)

!pip install -q datasets tokenizers
!wget -q https://raw.githubusercontent.com/krmhuseyn-sys/odlar/main/model.py -O model.py && echo "model.py OK"

In [ ]:
# 2) Data + tokenizer (Drive-da varsa, yenidən hazırlanmır). Təxminən 20-30 dəqiqə.
import numpy as np, time
from tokenizers import Tokenizer, ByteLevelBPETokenizer

TOK_PATH, TRAIN_PATH, VAL_PATH = f"{OUT}/tokenizer.json", f"{OUT}/train.npy", f"{OUT}/val.npy"

if all(os.path.exists(p) for p in (TOK_PATH, TRAIN_PATH, VAL_PATH)):
    print("Hazır data Drive-dan götürülür")
    tok = Tokenizer.from_file(TOK_PATH)
    train_data, val_data = np.load(TRAIN_PATH, mmap_mode="r"), np.load(VAL_PATH)
else:
    from datasets import load_dataset
    t0 = time.time()
    wiki, total = [], 0
    for t in load_dataset("wikimedia/wikipedia", "20231101.az", split="train")["text"]:
        t = t.strip()
        if len(t) < 200: continue
        wiki.append(t); total += len(t)
        if total >= WIKI_CHARS: break
    print(f"Vikipediya: {len(wiki):,} məqalə, {total/1e6:.0f}M simvol")

    web, total = [], 0
    for row in load_dataset("HuggingFaceFW/fineweb-2", name="azj_Latn", split="train", streaming=True):
        t = row["text"].strip()
        if len(t) < 300: continue
        web.append(t); total += len(t)
        if len(web) % 50_000 == 0: print(f"  FineWeb: {total/1e6:.0f}M simvol...")
        if total >= FINEWEB_CHARS: break
    print(f"FineWeb-2: {len(web):,} sənəd, {total/1e6:.0f}M simvol ({(time.time()-t0)/60:.1f} dəq)")

    def tok_iter():
        n = 0
        for t in wiki + web:
            yield t; n += len(t)
            if n >= TOK_TRAIN_CHARS: return
    btok = ByteLevelBPETokenizer()
    btok.train_from_iterator(tok_iter(), vocab_size=VOCAB_SIZE, min_frequency=2,
                             special_tokens=["<|endoftext|>"])
    btok.save(TOK_PATH)
    tok = Tokenizer.from_file(TOK_PATH)
    EOT = tok.token_to_id("<|endoftext|>")
    print(f"Tokenizer hazırdır ({(time.time()-t0)/60:.1f} dəq)")

    def encode(texts):
        parts = []
        for i in range(0, len(texts), 2000):
            for e in tok.encode_batch(texts[i:i+2000]):
                parts.append(np.array(e.ids + [EOT], dtype=np.uint16))
        return np.concatenate(parts)
    w, f = encode(wiki), encode(web); del wiki, web
    wv, fv = int(len(w) * 0.02), int(len(f) * 0.02)      # hər mənbədən 2% validasiya
    val_data = np.concatenate([w[-wv:], f[-fv:]])
    train_data = np.concatenate([w[:-wv], f[:-fv]]); del w, f
    np.save(TRAIN_PATH, train_data); np.save(VAL_PATH, val_data)
    print(f"Hazır: {(time.time()-t0)/60:.1f} dəq")

EOT = tok.token_to_id("<|endoftext|>")
print(f"Təlim: {len(train_data)/1e6:.0f}M token, validasiya: {len(val_data)/1e6:.1f}M token")
print("Nümunə:", tok.encode("Bakı Azərbaycanın paytaxtıdır.").tokens)

In [ ]:
# 3) Təlim (hər 500 addımda Drive-a yadda saxlayır, kəsilsə davam edir)
import math, time, torch
from dataclasses import asdict
from model import GPT, GPTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
use_amp = device == "cuda"
cfg = GPTConfig(vocab_size=tok.get_vocab_size(), block_size=BLOCK_SIZE,
                n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD)
model = GPT(cfg).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.1)
scaler = torch.amp.GradScaler(enabled=use_amp)

CKPT, BEST = f"{OUT}/checkpoint.pt", f"{OUT}/model.pt"
start, best = 0, float("inf")
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
    start, best = ck["step"] + 1, ck["best"]
    print(f"Qaldığı yerdən davam edir: addım {start} ")
print(f"Cihaz: {device} | Model: {model.num_params()/1e6:.1f} milyon parametr")

def get_batch(d):
    ix = np.random.randint(0, len(d) - BLOCK_SIZE - 1, BATCH_SIZE)
    x = torch.from_numpy(np.stack([d[i:i+BLOCK_SIZE] for i in ix]).astype(np.int64))
    y = torch.from_numpy(np.stack([d[i+1:i+1+BLOCK_SIZE] for i in ix]).astype(np.int64))
    return x.to(device), y.to(device)

def lr_at(step, warmup=300):
    if step < warmup:
        return LR * (step + 1) / warmup
    p = (step - warmup) / max(1, MAX_STEPS - warmup)
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0))))

@torch.no_grad()
def eval_loss(d, n=40):
    model.eval(); s = 0.0
    for _ in range(n):
        x, y = get_batch(d)
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
            s += model(x, y)[1].item()
    model.train(); return s / n

def sample(prompt, n=60):
    model.eval()
    idx = torch.tensor([tok.encode(prompt).ids], device=device)
    out = model.generate(idx, max_new_tokens=n, stop_id=EOT)[0].tolist()
    model.train(); return tok.decode(out)

t0 = time.time()
for step in range(start, MAX_STEPS + 1):
    for g in opt.param_groups:
        g["lr"] = lr_at(step)
    x, y = get_batch(train_data)
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=use_amp):
        _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()

    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        vl = eval_loss(val_data)
        el = (time.time()-t0)/60; done = step - start + 1
        eta = el / done * (MAX_STEPS - step) if done > 1 else 0
        print(f"addım {step:>6} | train {loss.item():.3f} | val {vl:.3f} | {el:.1f} dəq | qalıb ~{eta:.0f} dəq")
        if vl < best:
            best = vl
            torch.save({"config": asdict(cfg), "model": model.state_dict()}, BEST)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                    "step": step, "best": best}, CKPT)
        print("   →", sample(SAMPLE_PROMPT).replace("\n", " ")[:200])
print("Təlim bitdi! Ən yaxşı val loss:", round(best, 3))

In [ ]:
# 4) Ən yaxşı modeli sına
ck = torch.load(BEST, map_location=device, weights_only=True)
model = GPT(GPTConfig(**ck["config"])).to(device); model.load_state_dict(ck["model"])
for p in ["Bakı şəhəri", "Xəzər dənizi", "Nizami Gəncəvi", "Qədim zamanlarda"]:
    print(sample(p, 80), "\n---")

In [ ]:
# 5) Lokal sayt üçün zip (Drive-da: MyDrive/Odlar/my_llm.zip)
import zipfile, shutil
shutil.copy(TOK_PATH, "tokenizer.json"); shutil.copy(BEST, "model.pt")
zp = f"{OUT}/odlar15.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in ["model.pt", "tokenizer.json", "model.py"]:
        z.write(f)
print("Hazırdır:", zp, f"({os.path.getsize(zp)/1e6:.0f} MB)")